In [2]:
# Semantic Search using BM25, Sentence Transformers and RRF

## Objective

"""This notebook demonstrates:

- Keyword Search using BM25
- Semantic Search using Sentence Transformers
- Hybrid Search
- Reciprocal Rank Fusion (RRF)

The goal is to retrieve relevant documents using both keyword matching and semantic similarity."""

'This notebook demonstrates:\n\n- Keyword Search using BM25\n- Semantic Search using Sentence Transformers\n- Hybrid Search\n- Reciprocal Rank Fusion (RRF)\n\nThe goal is to retrieve relevant documents using both keyword matching and semantic similarity.'

In [3]:
!pip install -q sentence-transformers rank-bm25

In [4]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

In [ ]:
# Create Sample Knowledge Base

#The corpus acts as a collection of technical documents that users can search.

In [5]:
corpus = [
    "How to hard-reset your iPhone 13 if the touch screen is completely frozen or unresponsive.",
    "Troubleshooting guide for iOS updates failing on newer Apple mobile devices.",
    "The new Samsung Galaxy S26 Ultra features an advanced generative AI camera system.",
    "Steps to recover a lost Google Pixel account recovery phrase or authentication token.",
    "Fixing Wi-Fi connectivity drops and network configuration errors on Apple Macbook laptops.",
    "Why is my smartphone battery draining so quickly? Top power optimization tips."
]

print(f"Loaded database with {len(corpus)} technical documents.")

Loaded database with 6 technical documents.


In [ ]:
# BM25 Keyword Search

"""BM25 is a keyword-based retrieval algorithm.

It searches documents using exact word matching and ranks them based on relevance."""

In [6]:
tokenized_corpus = [doc.lower().split(" ") for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def sparse_search(query, top_n=5):

    tokenized_query = query.lower().split(" ")

    scores = bm25.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[::-1][:top_n]

    return [(idx, scores[idx]) for idx in top_indices if scores[idx] > 0]

In [ ]:
print("BM25 Search Result:")
print(sparse_search("iPhone 13"))

In [ ]:
# Semantic Search using Sentence Transformers

"""Unlike BM25, semantic search understands meaning.

It converts documents and queries into embeddings (vectors)."""

In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

In [ ]:
corpus_embeddings = embedding_model.encode(
    corpus,
    convert_to_tensor=True
)

In [ ]:
def dense_search(query, top_n=5):

    query_embedding = embedding_model.encode(
        query,
        convert_to_tensor=True
    )

    cos_scores = util.cos_sim(
        query_embedding,
        corpus_embeddings
    )[0]

    top_results = np.argsort(
        cos_scores.cpu().numpy()
    )[::-1][:top_n]

    return [
        (int(idx), float(cos_scores[idx]))
        for idx in top_results
    ]

In [ ]:
print("Semantic Search Result:")
print(dense_search("pixels in camera"))

In [ ]:
# Reciprocal Rank Fusion (RRF)

"""RRF combines rankings from multiple retrieval systems.

In this notebook we combine:

1. BM25 Search
2. Semantic Search

This improves retrieval accuracy."""

In [ ]:
def reciprocal_rank_fusion(
    sparse_results,
    dense_results,
    k=60,
    top_n=3
):

    rrf_scores = {}

    for rank, (doc_id, _) in enumerate(sparse_results):

        rrf_scores[doc_id] = (
            rrf_scores.get(doc_id, 0.0)
            + 1.0 / (k + rank + 1)
        )

    for rank, (doc_id, _) in enumerate(dense_results):

        rrf_scores[doc_id] = (
            rrf_scores.get(doc_id, 0.0)
            + 1.0 / (k + rank + 1)
        )

    fused_rankings = sorted(
        rrf_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

    return fused_rankings[:top_n]

In [8]:
# Hybrid Search Engine

"""Hybrid Search combines:

- BM25 keyword retrieval
- Semantic retrieval

This architecture is commonly used in RAG systems."""

'Hybrid Search combines:\n\n- BM25 keyword retrieval\n- Semantic retrieval\n\nThis architecture is commonly used in RAG systems.'

In [ ]:
def hybrid_search_engine(query, top_n=3):

    sparse_res = sparse_search(
        query,
        top_n=10
    )

    dense_res = dense_search(
        query,
        top_n=10
    )

    hybrid_res = reciprocal_rank_fusion(
        sparse_res,
        dense_res,
        k=60,
        top_n=top_n
    )

    print(f"\nQuery: {query}")

    for rank, (doc_id, score) in enumerate(hybrid_res):

        print("\n--------------------------------")

        print(f"Rank: {rank + 1}")

        print(f"RRF Score: {score:.4f}")

        print(f"Document: {corpus[doc_id]}")

In [ ]:
# Test Case 1

#Query contains Apple-related device issues.

#This demonstrates how semantic search can identify relevant documents even when exact keywords are missing.

In [ ]:
hybrid_search_engine(
    "Apple mobile device issues"
)

In [ ]:
# Test Case 2

"""Query combines multiple concepts:

- Macbook
- Battery Optimization

This demonstrates the effectiveness of Hybrid Search."""

In [ ]:
hybrid_search_engine(
    "Macbook laptop battery optimization"
)

In [ ]:
#Ground Truth Dataset
ground_truth = {
    "Apple mobile device issues": [1],
    "pixels in camera": [2],
    "Macbook laptop battery optimization": [4, 5]
}

In [ ]:
#MRR Function

def calculate_mrr(ground_truth):

    reciprocal_ranks = []

    for query, relevant_docs in ground_truth.items():

        sparse_res = sparse_search(query, top_n=10)
        dense_res = dense_search(query, top_n=10)

        hybrid_res = reciprocal_rank_fusion(
            sparse_res,
            dense_res,
            top_n=10
        )

        ranked_docs = [doc_id for doc_id, _ in hybrid_res]

        rr = 0

        for rank, doc_id in enumerate(ranked_docs, start=1):

            if doc_id in relevant_docs:
                rr = 1 / rank
                break

        reciprocal_ranks.append(rr)

    return np.mean(reciprocal_ranks)

In [ ]:
#Import NDCG Library
from sklearn.metrics import ndcg_score

In [ ]:
#NDCG Function
def calculate_ndcg(ground_truth):

    ndcg_scores = []

    for query, relevant_docs in ground_truth.items():

        sparse_res = sparse_search(query, top_n=10)
        dense_res = dense_search(query, top_n=10)

        hybrid_res = reciprocal_rank_fusion(
            sparse_res,
            dense_res,
            top_n=10
        )

        ranked_docs = [doc_id for doc_id, _ in hybrid_res]

        y_true = [
            1 if doc in relevant_docs else 0
            for doc in ranked_docs
        ]

        y_score = list(
            range(len(ranked_docs), 0, -1)
        )

        score = ndcg_score(
            [y_true],
            [y_score]
        )

        ndcg_scores.append(score)

    return np.mean(ndcg_scores)

In [ ]:
#Run Evaluation

mrr = calculate_mrr(ground_truth)

ndcg = calculate_ndcg(ground_truth)

print("=" * 40)
print("SEARCH EVALUATION RESULTS")
print("=" * 40)

print(f"MRR Score  : {mrr:.4f}")
print(f"NDCG Score : {ndcg:.4f}")

In [ ]:
# Search Evaluation Metrics
"""
This section evaluates the performance of the Hybrid Search Engine using:

- MRR (Mean Reciprocal Rank)
- NDCG (Normalized Discounted Cumulative Gain)

These metrics help measure retrieval effectiveness and ranking quality."""